# Notebook Training Model di Kaggle biar ga lamaaaaa


In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch import nn, optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

print("PyTorch version:", torch.__version__)
print("GPU tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nama GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Sesuaikan path dengan lokasi dataset di Kaggle
# Biasanya letak dataset setelah ditambahkan di sebelah kanan ada di path ini:
data_dir = "/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"

# Jika terjadi Error FileNotFoundError, Anda bisa menjalankan cell ini untuk memastikan nama foldernya benar:
# import glob
# print(glob.glob('/kaggle/input/new-plant-diseases-dataset/*/*'))

# Cek apakah folder sudah ada isinya
for folder in os.listdir(data_dir):
    print(folder)

In [ ]:
# Transformasi gambar — resize, augmentasi, ubah ke tensor
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),       # Samakan ukuran semua foto
    transforms.RandomHorizontalFlip(),   # Flip acak
    transforms.RandomRotation(10),       # Rotasi acak sedikit
    transforms.ToTensor(),               # Ubah gambar jadi angka
    transforms.Normalize([0.485, 0.456, 0.406],  # Normalisasi warna
                         [0.229, 0.224, 0.225])
])

valid_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load dataset dari folder
train_data = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transforms)
valid_data = datasets.ImageFolder(os.path.join(data_dir, "valid"), transform=valid_transforms)

# PENTING: Gunakan num_workers=2 di Kaggle agar mempercepat transfer data, batch size dinaikkan ke 64
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_data, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print("Jumlah foto training :", len(train_data))
print("Jumlah foto validasi :", len(valid_data))
print("Jumlah kelas penyakit:", len(train_data.classes))


In [ ]:
# Load ResNet18 pre-trained
model = models.resnet18(weights="IMAGENET1K_V1")

# Bekukan semua layer lama
for param in model.parameters():
    param.requires_grad = False

# Ganti layer terakhir
jumlah_kelas = len(train_data.classes)
model.fc = nn.Linear(model.fc.in_features, jumlah_kelas)

# Pakai GPU kalau ada, kalau tidak pakai CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model siap! Jumlah kelas:", jumlah_kelas)
print("Training di:", device)

In [ ]:
# Fungsi loss dan optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training loop
EPOCHS = 5  # Di Kaggle (GPU), 5 epoch akan berjalan sangat cepat!
train_losses, valid_losses, valid_accs = [], [], []

for epoch in range(EPOCHS):
    # === TRAINING ===
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # === VALIDASI ===
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            val_loss += criterion(output, labels).item()
            _, predicted = torch.max(output, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    acc = correct / total * 100
    train_losses.append(running_loss / len(train_loader))
    valid_losses.append(val_loss / len(valid_loader))
    valid_accs.append(acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {running_loss/len(train_loader):.3f} | "
          f"Val Loss: {val_loss/len(valid_loader):.3f} | "
          f"Akurasi: {acc:.1f}%")

print("\nTraining selesai!")

In [ ]:
# Visualisasi Grafik Loss dan Akurasi
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(valid_losses, label='Valid Loss', marker='o')
plt.title('Grafik Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Akurasi
plt.subplot(1, 2, 2)
plt.plot(valid_accs, label='Valid Akurasi', marker='o', color='green')
plt.title('Grafik Akurasi Validasi per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Akurasi (%)')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Simpan model ke direktori working Kaggle
# Direktori /kaggle/working adalah folder tempat file yang bisa Anda download
model_path = "/kaggle/working/plant_model.pth"
torch.save(model.state_dict(), model_path)
print(f"Model berhasil disimpan di: {model_path}\nAnda bisa mendownloadnya dari panel 'Output' di sebelah kanan Kaggle.")